# Exercise 4 - RAG: retrieve, then generate (slides 18-25)

This notebook runs **inside IRIS**: its Python kernel is IRIS's own Python, so `import iris` talks to the database
directly - no driver, no connection string. The SQL below is the same SQL you ran in the Management Portal.

**Connect the notebook (once):** in VS Code, click **Select Kernel** (top right) -> **Select Another Kernel...** ->
**Existing Jupyter Server...** -> paste `http://127.0.0.1:48888/?token=workshop` -> pick **Python 3 (IRIS)**.

Run the cells top to bottom with **Shift+Enter**: every step shows its result before the next one uses it.
Two small TODOs (3 and 4) - paste the line from the TODO comment - - the same as in `Workshop.Exercise.Ex4`, if you prefer ObjectScript (last section).
Stuck? The completed notebook is `solutions/ex4_rag.ipynb`.

In [ ]:
# Step 1: paste the workshop's OpenAI key (shared, valid today only) between the quotes, then run this cell.
# It is stored in IRIS (Workshop.Util.SetKey) - the ObjectScript option uses the same key. Run once; you can clear it afterwards.
OPENAI_KEY = ""

import json, html
import iris
from IPython.display import HTML, display

util = iris.cls("Workshop.Util")
if OPENAI_KEY:
    print(util.SetKey(OPENAI_KEY))
# Ready check: model downloaded (Exercise 1), chunks from your Exercise 2, the key.
SYSTEM_PROMPT = util.SystemPrompt()    # data/prompts/system_prompt.txt
print(util.Checkpoint(4))      # catches up if needed: model + registration (Ex 1), chunks with YOUR Ex 2 code
print("LLM:", util.LlmStatus())

## 1. Retrieve: which passages fit the question?

The same search as in Exercise 2: the question becomes a vector with the **same model** as the chunks
(`EMBEDDING(?)`), `VECTOR_DOT_PRODUCT` scores every chunk, and the best `k` come back. In Python each result row
becomes a small dictionary - a **hit**. Run the cell and look at what one hit contains.

In [ ]:
QUESTION = "How long do we keep lab backups?"

SEARCH_SQL = '''
SELECT TOP ? chunk_id, title, source, content,
       VECTOR_DOT_PRODUCT(embedding, EMBEDDING(?)) AS score
FROM Workshop.SupportChunks
ORDER BY score DESC'''

def search(question, k=3, min_score=None):
    rows = iris.sql.exec(SEARCH_SQL, k, question)
    hits = [{"chunk_id": r[0], "title": r[1], "source": r[2], "content": r[3], "score": float(r[4])} for r in rows]
    if min_score is not None:
        hits = [h for h in hits if h["score"] >= min_score]
    return hits

hits = search(QUESTION)
print(len(hits), "hits. The first one:\n")
for key, value in hits[0].items():
    print(f"  {key:9} {value if key != 'content' else value[:100] + ' ...'}")
print("\nAll hits, best first:")
for h in hits:
    print(f"  {h['score']:.3f}  {h['chunk_id']:<20} {h['title']}")

**What you see:** `hits` is a list (best first) and every hit has the same five keys: `chunk_id` (e.g.
`backup-restore §2` = backup & restore handbook, section 2), `title`, `source` (the PDF), `content` (the text of the
section) and `score` (the dot product with the question - higher = closer in meaning).

## 2. Turn the passages into text for the LLM (TODO 3)

An LLM only reads text. So every hit becomes one block: **the chunk ID in brackets, the title and the source on the
first line, then the passage**. The ID in brackets matters: the LLM will quote it to say *where* its answer comes from.
For the hit above the block must look like this:

```
[backup-restore §2] Backup schedule and retention (source: backup-restore-handbook.pdf)
Every environment is backed up automatically each night at 02:00 UTC. ...
```

**TODO 3:** paste the line from the TODO comment into `passage()`. It builds exactly that text from the keys of step 1
(`hit["chunk_id"]`, `hit["title"]`, `hit["source"]`, `hit["content"]`; `\n` starts the second line). Run the cell - it
prints your block.

In [ ]:
def passage(hit):
    # TODO 3: Paste this line: return f"[{hit['chunk_id']}] {hit['title']} (source: {hit['source']})\n{hit['content']}"
    raise NotImplementedError("TODO 3 is not implemented yet. " + 'Paste this line: return f"[{hit[\'chunk_id\']}] {hit[\'title\']} (source: {hit[\'source\']})\\n{hit[\'content\']}"')

print(passage(hits[0]))

The context is simply all blocks, separated by a blank line (given):

In [ ]:
def build_context(hits):
    return "\n\n".join(passage(h) for h in hits)

context = build_context(hits)
print(context)

## 3. The message to the LLM

The LLM gets two texts: the **system prompt** (the rules: answer only from the passages, cite the chunk IDs, say so
when the passages do not answer, reply as JSON) and the **user message** (the passages + the question). Read both -
this is everything the LLM will know.

In [ ]:
print("SYSTEM PROMPT:\n" + SYSTEM_PROMPT)

user_msg = "Passages:\n\n" + context + "\n\nQuestion: " + QUESTION
print("\nUSER MESSAGE:\n" + user_msg)

## 4. Ask the LLM (generate)

`util.CompleteJSON` is IRIS code: an HTTPS request from IRIS to OpenAI with the key from step 1. The reply is JSON
with three fields: `sufficient_evidence` (did the passages answer the question?), `answer`, and `cited_chunk_ids`
(which passages it used).

In [ ]:
reply = json.loads(util.CompleteJSON(SYSTEM_PROMPT, user_msg))
print(json.dumps(reply, indent=2, ensure_ascii=False))

## 5. Check the citations (TODO 4)

Never trust a citation blindly: an LLM can cite a chunk it was never given (or invent an ID). So we compare: which
IDs in `reply["cited_chunk_ids"]` are **not** among the `chunk_id`s of our hits? For a good answer that list is
empty.

**TODO 4:** paste the line from the TODO comment into `unknown_citations()`. `retrieved` is the list of IDs we gave the
LLM; the line returns the cited IDs that are not in it. The cell then tests it twice: with the real reply (expect `[]`), and with a made-up reply that
cites a chunk we never retrieved (expect `['made-up §9']`).

In [ ]:
def unknown_citations(reply, hits):
    retrieved = [h["chunk_id"] for h in hits]      # the IDs we gave the LLM
    # TODO 4: Paste this line: return [c for c in reply["cited_chunk_ids"] if c not in retrieved]
    raise NotImplementedError("TODO 4 is not implemented yet. " + 'Paste this line: return [c for c in reply["cited_chunk_ids"] if c not in retrieved]')

print("real reply:   ", unknown_citations(reply, hits))
print("made-up reply:", unknown_citations({"cited_chunk_ids": ["backup-restore §2", "made-up §9"]}, hits))

## 6. All together: `ask()` - retrieve, then generate

Steps 1-5 in one function (given), and `show()` to display the answer next to its evidence (cited passages are
marked). From here on you just ask questions.

In [ ]:
def ask(question, k=3, min_score=None):
    hits = search(question, k, min_score)                                   # 1. retrieve
    context = build_context(hits) or "(no passages retrieved)"              # 2. passages as text
    user_msg = "Passages:\n\n" + context + "\n\nQuestion: " + question      # 3. the message
    reply = json.loads(util.CompleteJSON(SYSTEM_PROMPT, user_msg))          # 4. generate
    if "error" in reply:
        raise RuntimeError(reply["error"])
    return {"question": question, "hits": hits, "prompt": user_msg, "reply": reply,
            "unknown_citations": unknown_citations(reply, hits)}            # 5. check the citations


def show(result, show_prompt=False):
    """The answer next to its evidence: cited passages are marked."""
    if "error" in result:
        display(HTML(f"<b style='color:#b00'>Error:</b> {html.escape(result['error'])}"))
        return
    r = result["reply"]
    cited = set(r.get("cited_chunk_ids", []))
    color = "#0a7d32" if r.get("sufficient_evidence") else "#a15c00"
    rows = "".join(
        f"<tr><td>{'&#10004;' if h['chunk_id'] in cited else ''}</td><td>{h['score']:.3f}</td>"
        f"<td><code>{html.escape(h['chunk_id'])}</code></td><td>{html.escape(h['content'][:160])}...</td></tr>"
        for h in result["hits"])
    warn = (f"<p style='color:#b00'>Cited but not retrieved: {result['unknown_citations']}</p>"
            if result["unknown_citations"] else "")
    page = (f"<p><b>Q:</b> {html.escape(result['question'])}</p>"
            f"<p style='border-left:4px solid {color};padding-left:8px'><b>A:</b> {html.escape(r.get('answer', ''))}</p>"
            f"<p style='color:{color}'>sufficient_evidence = {r.get('sufficient_evidence')} &nbsp; "
            f"<small>({r.get('model', '')}, {r.get('latency_ms', 0):.0f} ms)</small></p>{warn}"
            f"<table><tr><th>cited</th><th>score</th><th>chunk</th><th>passage</th></tr>{rows}</table>")
    if show_prompt:
        page += f"<details open><summary>Prompt sent to the LLM</summary><pre>{html.escape(result['prompt'])}</pre></details>"
    display(HTML(page))


show(ask("How long do we keep lab backups?"))

## 7. Test it: read each answer next to its evidence

* a question the documents answer, asked in other words (paraphrase),
* a question the documents do **not** answer - a good RAG system says so instead of guessing.

In [ ]:
paraphrase = ask("For how many days are the nightly snapshots of a lab environment retained?")
unsupported = ask("What uptime percentage does the platform guarantee in its SLA?")
show(paraphrase)
show(unsupported)

## 8. Add knowledge without retraining: the runbook (failed-imports §1)

Ask, add one PDF to the knowledge base (with **your** Exercise 2 code), ask again. Nothing about the model changes -
only the data in IRIS. At the end the runbook is removed again, so you can repeat this.

In [ ]:
question = "Where do failed imports go in our environment, and how long are they kept?"
before = ask(question)
show(before)
print("ingested chunks:", util.IngestRunbook())
try:
    after = ask(question)
    show(after)
finally:
    util.RemoveRunbook()             # always: the next run starts without the runbook again
    print("runbook removed again")

In [ ]:
# Success check (same checks as Workshop.Exercise.Ex4.Run)
a, u, b, f = ask("How long do we keep lab backups?")["reply"], unsupported["reply"], before["reply"], after["reply"]
assert passage(search("How long do we keep lab backups?")[0]).startswith("[backup-restore §2] "), "passage() should start with [chunk_id]"
assert unknown_citations({"cited_chunk_ids": ["made-up §9"]}, hits) == ["made-up §9"], "unknown_citations() should find made-up IDs"
assert a["sufficient_evidence"] and "14" in a["answer"], "answerable question should say 14 days"
assert not u["sufficient_evidence"] and not u["cited_chunk_ids"], "unsupported question should cite nothing"
assert not b["sufficient_evidence"], "before ingestion there is no evidence"
assert "failed-imports §1" in f["cited_chunk_ids"] and "11" in f["answer"], "after ingestion: cite failed-imports §1 and say 11 days"
assert not after["unknown_citations"], "citations must be retrieved chunks"
print("EXERCISE 4 PASSED")

## 9. Experiment

Change the settings and run the cell again:

* `K` - how many passages the LLM gets. Try 1 and 8.
* `MIN_SCORE` - drop weak passages (e.g. `0.3`). What happens to the SLA question?
* `SHOW_PROMPT` - see exactly what is sent to the LLM.

In [ ]:
QUESTION = "My password reset email never showed up. What should I do?"
K = 3
MIN_SCORE = None        # e.g. 0.3
SHOW_PROMPT = True

show(ask(QUESTION, K, MIN_SCORE), show_prompt=SHOW_PROMPT)

## 10. The ObjectScript option

If you did TODO 3 and 4 in `Workshop.Exercise.Ex4` (VS Code) instead, run your ObjectScript code from here.
Same pipeline, same result - Python and ObjectScript call the same IRIS.

In [ ]:
def ask_objectscript(question, k=3, min_score=None):
    res = json.loads(iris.cls("Workshop.Exercise.Ex4").AskJSON(question, k, "" if min_score is None else min_score))
    if "error" in res:
        return res
    return {"question": question, "hits": res["retrieval"]["hits"], "prompt": "(built in ObjectScript)",
            "reply": res["reply"], "unknown_citations": res["unknown_citations"]}

show(ask_objectscript("How long do we keep lab backups?"))